# Week 09. Annotated, Query 모델, 검증 규칙

9주차는 요청 파라미터를 더 엄격하게 검증하고, 여러 쿼리 조건을 하나의 모델로 묶는 방법을 다룹니다.
FastAPI 없이도 Pydantic 모델을 통해 같은 검증 개념을 실행합니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. 검색 조건 모델 만들기

여러 쿼리 파라미터를 함수 인자로 흩뿌리면 관리가 어려워집니다.
모델로 묶으면 기본값과 검증 규칙을 한 곳에서 관리할 수 있습니다.


In [1]:
from typing import Annotated

from pydantic import BaseModel, Field, ValidationError


class SearchQuery(BaseModel):
    keyword: Annotated[str, Field(min_length=2, max_length=30)]
    page: Annotated[int, Field(ge=1)] = 1
    size: Annotated[int, Field(ge=1, le=50)] = 10
    tags: list[str] = Field(default_factory=list)


query = SearchQuery(keyword="python", page=1, size=5, tags=["api", "data"])
query


SearchQuery(keyword='python', page=1, size=5, tags=['api', 'data'])

## 2. 검증 실패를 명확히 확인

잘못된 요청을 조용히 통과시키면 API 내부에서 더 큰 오류가 납니다.
요청 경계에서 빨리 실패시키는 것이 안정적입니다.


In [2]:
invalid_cases = [
    {"keyword": "p", "page": 1, "size": 10},
    {"keyword": "python", "page": 0, "size": 10},
    {"keyword": "python", "page": 1, "size": 100},
]

for payload in invalid_cases:
    try:
        SearchQuery(**payload)
    except ValidationError as error:
        print(payload, "->", error.errors()[0]["type"])


{'keyword': 'p', 'page': 1, 'size': 10} -> string_too_short
{'keyword': 'python', 'page': 0, 'size': 10} -> greater_than_equal
{'keyword': 'python', 'page': 1, 'size': 100} -> less_than_equal


## 3. 검색 함수에 모델 적용하기

검증된 모델을 함수에 넘기면 함수 내부에서는 값이 정상 범위라고 믿고 로직에 집중할 수 있습니다.


In [3]:
articles = [
    {"title": "Python API Basics", "tags": ["python", "api"]},
    {"title": "Data Cleaning Note", "tags": ["python", "data"]},
    {"title": "Frontend Memo", "tags": ["web"]},
    {"title": "Async Python", "tags": ["python", "async"]},
]


def search_articles(params: SearchQuery):
    keyword = params.keyword.lower()
    matched = [
        article
        for article in articles
        if keyword in article["title"].lower()
        and (not params.tags or any(tag in article["tags"] for tag in params.tags))
    ]
    start = (params.page - 1) * params.size
    end = start + params.size
    return matched[start:end]


search_articles(SearchQuery(keyword="python", tags=["api"]))


[{'title': 'Python API Basics', 'tags': ['python', 'api']}]

## 정리

- `Annotated`와 `Field`를 조합하면 타입과 검증 규칙을 한 위치에서 표현할 수 있습니다.
- 쿼리 모델은 검색 조건이 많아질수록 유용합니다.
- 입력 검증을 통과한 뒤에는 비즈니스 로직이 단순해집니다.
